# Taller Autoencoders - Clasificación de objetos (MLP sobre latentes del encoder)
Big Data - MCIC

Pillt Hernandez

Cod 20242595003

## Preparación del ambiente
- Se cargan librerías necesarias
- Se conecta con Drive

In [ ]:
import os, numpy as np, tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
# Conexión con Drive para permitir acceso al dataset
from google.colab import drive
drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders"

## Carga y preparación de latentes y etiquetas

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hiperparámetros
BATCH = 16          # entrenar sobre latentes es barato → se puede subir
EPOCHS = 80
VAL_FRAC = 0.1

# --- 1) Cargar latentes si existen; si no, generarlos con el encoder ---
z_train_path = os.path.join(drive_folder, "z_train.npy")
z_test_path = os.path.join(drive_folder, "z_test.npy")
encoder_path = os.path.join(drive_folder, "encoder.keras")
y_train_path = os.path.join(drive_folder, "y_train.npy")
y_test_path = os.path.join(drive_folder, "y_test.npy")


if os.path.exists(z_train_path) and os.path.exists(z_test_path):
    print("Cargando latentes desde disco...")
    z_train = np.load(z_train_path)
    z_test  = np.load(z_test_path)
else:
    print("No hay latentes guardados. Cargando encoder y calculando z...")
    encoder = tf.keras.models.load_model(encoder_path, compile=False)
    z_train = encoder.predict(x_train, batch_size=8, verbose=0)
    z_test  = encoder.predict(x_test,  batch_size=8, verbose=0)

print("z_train:", z_train.shape, "z_test:", z_test.shape)

# --- Load labels ---
if os.path.exists(y_train_path) and os.path.exists(y_test_path):
    print("Cargando etiquetas desde disco...")
    y_train = np.load(y_train_path)
    y_test = np.load(y_test_path)
else:
    print("No se encontraron archivos de etiquetas. Asegúrate de que y_train.npy y y_test.npy existan en la carpeta de Drive.")
    # You might want to add code here to load or generate y_train and y_test
    # if they don't exist. For now, I'll just print a message.


# --- 2) Codificar etiquetas (string → entero) ---
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
num_classes = len(le.classes_)
LATENT_DIM  = z_train.shape[1]

print("Conteos TRAIN:", Counter(y_train))
print("Conteos TEST :", Counter(y_test))
print("Dim. latente (z_train) =", z_train.shape[1])


# --- 3) Split de validación (estratificado) desde z_train ---
z_tr, z_val, y_tr, y_val = train_test_split(
    z_train, y_train_enc,
    test_size=VAL_FRAC, random_state=SEED, stratify=y_train_enc
)

print("Split latentes → z_tr:", z_tr.shape, "z_val:", z_val.shape)

## Definición del clasificador (MLP) y entrenamiento

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# --- 4) MLP sobre el vector latente ---
inp = layers.Input(shape=(LATENT_DIM,))
x = layers.Dense(128, activation="relu")(inp)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
out = layers.Dense(num_classes, activation="softmax")(x)

clf = tf.keras.Model(inp, out, name="latent_mlp_classifier")
clf.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# --- 5) Pesos por clase para mitigar desbalance ---
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_tr
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print("Pesos por clase:", {le.classes_[i]: float(w) for i, w in class_weight_dict.items()})

# --- 6) Callbacks ---
es  = EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True)
rlr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1)


In [ ]:
# --- 7) Entrenamiento ---
hist = clf.fit(
    z_tr, y_tr,
    validation_data=(z_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    class_weight=class_weight_dict,
    callbacks=[es, rlr],
    verbose=1
)

## Curvas de entrenamiento (loss / accuracy)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist.history["loss"], label="train")
plt.plot(hist.history["val_loss"], label="val")
plt.title("Pérdida (SCCE)")
plt.xlabel("Época"); plt.ylabel("Loss"); plt.legend()

plt.subplot(1,2,2)
plt.plot(hist.history["accuracy"], label="train")
plt.plot(hist.history["val_accuracy"], label="val")
plt.title("Exactitud")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.legend()

plt.tight_layout()
plt.show()


## Evaluación en TEST
- Accuracy, F1, reporte y matriz de confusión

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns  # si no lo tienes, puedes usar sólo matplotlib

# --- 8) Predicción en test ---
y_prob = clf.predict(z_test, batch_size=BATCH, verbose=0)
y_pred = y_prob.argmax(axis=1)

acc = accuracy_score(y_test_enc, y_pred)
f1_macro = f1_score(y_test_enc, y_pred, average="macro")

print(f"Accuracy (TEST): {acc:.4f}")
print(f"F1-macro  (TEST): {f1_macro:.4f}\n")

print("=== Classification Report (TEST) ===")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_, digits=3))

# --- 9) Matriz de confusión (normalizada por verdaderos) ---
cm = confusion_matrix(y_test_enc, y_pred, labels=np.arange(num_classes))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6.5,5.5))
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=le.classes_, yticklabels=le.classes_,
            cbar=False, square=True)
plt.title("Matriz de confusión (normalizada)")
plt.xlabel("Predicción"); plt.ylabel("Verdadero")
plt.tight_layout()
plt.show()


## Guardar el clasificador y artefactos

In [ ]:
import pickle
from pathlib import Path
save_dir = Path("/content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders")

SEED = 42
run_id = f"d{LATENT_DIM}_s{SEED}"

clf.save(save_dir / f"classifier_{run_id}.keras")
np.save(save_dir / "label_classes.npy", le.classes_)  # suficiente para reconstruir el LabelEncoder

# (Opcional) si quieres el objeto LabelEncoder tal cual:
with open(save_dir / f"label_encoder_{run_id}.pkl", "wb") as f:
    pickle.dump(le, f)

print(f"Guardado: classifier_{run_id}.keras, label_classes.npy (+ opcional pkl)")